In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Dirección de los datasets en GitHub
url = "https://raw.githubusercontent.com/fabri-mati-gonza/Trabrajo-IPC-FINAL/main/Data/"

# Cargamos los cuatro datasets
clubs = pd.read_csv(url + "Info%20de%20los%20clubes.gz", compression="gzip")
competitions = pd.read_csv(url + "Info%20de%20Ligas.gz", compression="gzip")
partidos = pd.read_csv(url + "Info%20de%20los%20partidos.gz", compression="gzip")
Datos_de_jugadores = pd.read_csv(url + "Info%20de%20los%20jugadores.gz", compression="gzip")

# Comprobamos que se cargaron correctamente
print("Clubes:", clubs.shape)
print("Competiciones:", competitions.shape)
print("Partidos:", partidos.shape)
print("Valores de jugadores:", Datos_de_jugadores.shape)

In [ ]:
# Mostramos las primeras cinco filas del dataset de clubes
clubs.head()

In [ ]:
# Mostramos las primeras cinco filas
competitions.head()

In [ ]:
# Mostramos las primeras cinco filas
partidos.head()

In [ ]:
# Mostramos las primeras cinco filas
Datos_de_jugadores.head()

In [ ]:
# Analizamos la estructura de los cuatro datasets
print("CLUBES:")
clubs.info()
print("\nCOMPETICIONES:")
competitions.info()
print("\nPARTIDOS:")
partidos.info()
print("\nVALORES DE MERCADO:")
Datos_de_jugadores.info()

In [ ]:
# Analizamos las estadísticas de los cuatro datasets
print("CLUBES:")
display(clubs.describe())
print("COMPETICIONES:")
display(competitions.describe())
print("PARTIDOS:")
display(partidos.describe())
print("VALORES DE MERCADO:")
display(Datos_de_jugadores.describe())

In [ ]:
# Analizamos los valores nulos de nuestros datasets
print("VALORES NULOS DE CLUBES:")
print(clubs.isna().sum())
print("\nVALORES NULOS DE COMPETICIONES:")
print(competitions.isna().sum())
print("\nVALORES NULOS DE PARTIDOS:")
print(partidos.isna().sum())
print("\nVALORES NULOS DE JUGADORES:")
print(Datos_de_jugadores.isna().sum())

In [ ]:
# Analizamos los registros duplicados de los cuatro datasets
print("DUPLICADOS DE CLUBES:", clubs.duplicated().sum())
print("DUPLICADOS DE COMPETICIONES:", competitions.duplicated().sum())
print("DUPLICADOS DE PARTIDOS:", partidos.duplicated().sum())
print("DUPLICADOS DE JUGADORES:", Datos_de_jugadores.duplicated().sum())

In [ ]:
# Filtramos los partidos de las temporadas elegidas
partidos_filtrados = partidos[partidos["season"].isin([2021, 2022, 2023])].copy()

# Mostramos cuántos partidos tenemos por temporada
print("Cantidad total de partidos:", partidos_filtrados.shape[0])
print("\nPartidos por temporada:")
print(partidos_filtrados["season"].value_counts().sort_index())

In [ ]:
# Filtramos solamente los partidos de Champions League
champions = partidos_filtrados[partidos_filtrados["competition_id"] == "CL"].copy()

# Contamos los partidos de Champions por temporada
print("TOTAL DE PARTIDOS DE CHAMPIONS:", champions.shape[0])
print("\nPARTIDOS DE CHAMPIONS POR TEMPORADA:")
print(champions["season"].value_counts().sort_index())

# Mostramos los primeros cinco partidos
champions.head()

In [ ]:
# Identificamos los clubes que jugaron la Champions

# Equipos que jugaron como locales
equipos_locales = champions[["season", "home_club_id"]].copy()
equipos_locales = equipos_locales.rename(columns={"home_club_id": "club_id"})

# Equipos que jugaron como visitantes
equipos_visitantes = champions[["season", "away_club_id"]].copy()
equipos_visitantes = equipos_visitantes.rename(columns={"away_club_id": "club_id"})

# Unimos los equipos locales y visitantes
equipos_champions = pd.concat([equipos_locales, equipos_visitantes])

# Eliminamos los registros repetidos
equipos_champions = equipos_champions.drop_duplicates()

# Mostramos la cantidad de equipos por temporada
print("EQUIPOS PARTICIPANTES POR TEMPORADA:")
print(equipos_champions.groupby("season")["club_id"].nunique())

# Mostramos las primeras cinco filas
equipos_champions.head()

In [ ]:
# Combinamos los equipos de Champions con sus nombres y ligas

equipos_champions = equipos_champions.merge(clubs[["club_id", "name", "domestic_competition_id"]],on="club_id", how="left")

# Mostramos los primeros diez equipos
equipos_champions.head(10)

In [ ]:
# Verificamos que el merge se haya realizado correctamente

print("TOTAL DE REGISTROS:", equipos_champions.shape[0])
print("EQUIPOS SIN NOMBRE:", equipos_champions["name"].isna().sum())
print("EQUIPOS SIN LIGA:", equipos_champions["domestic_competition_id"].isna().sum())
print("REGISTROS CLUB-TEMPORADA DUPLICADOS:",equipos_champions.duplicated(subset=["season", "club_id"]).sum())

In [ ]:
# Identificamos los equipos que no encontraron coincidencia
equipos_faltantes = equipos_champions[equipos_champions["name"].isna()]

# Mostramos los equipos faltantes
equipos_faltantes

In [ ]:
# Buscamos los nombres de los equipos faltantes en los partidos

# Equipos locales
nombres_locales = champions[["season", "home_club_id", "home_club_name"]].copy()
nombres_locales = nombres_locales.rename(columns={"home_club_id": "club_id","home_club_name": "nombre"})

# Equipos visitantes
nombres_visitantes = champions[["season", "away_club_id", "away_club_name"]].copy()
nombres_visitantes = nombres_visitantes.rename(columns={"away_club_id": "club_id","away_club_name": "nombre"})

# Unimos la información y buscamos los equipos faltantes
nombres_equipos = pd.concat([nombres_locales, nombres_visitantes])
nombres_equipos = nombres_equipos.drop_duplicates()
nombres_equipos[nombres_equipos["club_id"].isin([2481, 1064])]

In [ ]:
# Recuperamos los nombres desde el dataset de partidos
nombres_unicos = nombres_equipos.drop_duplicates(subset=["season", "club_id"])

# Combinamos la información
equipos_champions = equipos_champions.merge(nombres_unicos,on=["season", "club_id"],how="left")

# Completamos los nombres faltantes
equipos_champions["name"] = equipos_champions["name"].fillna(equipos_champions["nombre"])

# Eliminamos la columna auxiliar
equipos_champions = equipos_champions.drop(columns=["nombre"])

# Verificamos los resultados
print("TOTAL DE REGISTROS:", equipos_champions.shape[0])
print("EQUIPOS SIN NOMBRE:", equipos_champions["name"].isna().sum())
print("EQUIPOS SIN LIGA:", equipos_champions["domestic_competition_id"].isna().sum())

In [ ]:
# Seleccionamos los identificadores y nombres de las competiciones
ligas = competitions[["competition_id", "name"]].copy()

# Renombramos las columnas para relacionarlas con nuestros equipos
ligas = ligas.rename(columns={"competition_id": "domestic_competition_id","name": "nombre_liga"})

# Combinamos los equipos con sus respectivas ligas
equipos_con_liga = equipos_champions.merge(ligas,on="domestic_competition_id",how="left")

# Mostramos los resultados
print("TOTAL DE REGISTROS:", equipos_con_liga.shape[0])
print("EQUIPOS SIN LIGA:", equipos_con_liga["nombre_liga"].isna().sum())
equipos_con_liga.head(10)

In [ ]:
# Identificamos las competiciones que son ligas nacionales
ligas_nacionales = competitions[competitions["type"] == "domestic_league"]

# Filtramos los partidos de esas ligas
partidos_liga = partidos_filtrados[partidos_filtrados["competition_id"].isin(ligas_nacionales["competition_id"])].copy()

# Mostramos los resultados
print("TOTAL DE PARTIDOS DE LIGA:", partidos_liga.shape[0])
print("\nPARTIDOS DE LIGA POR TEMPORADA:")
print(partidos_liga["season"].value_counts().sort_index())
print("\nCANTIDAD DE LIGAS:", partidos_liga["competition_id"].nunique())

In [ ]:
# Separamos los equipos locales
locales_liga = partidos_liga[["season", "competition_id", "home_club_id"]].copy()
locales_liga = locales_liga.rename(columns={"competition_id": "domestic_competition_id","home_club_id": "club_id"})

# Separamos los equipos visitantes
visitantes_liga = partidos_liga[["season", "competition_id", "away_club_id"]].copy()

visitantes_liga = visitantes_liga.rename(columns={"competition_id": "domestic_competition_id","away_club_id": "club_id"})

# Unimos ambas tablas
equipos_liga = pd.concat([locales_liga, visitantes_liga])

# Contamos partidos por equipo, liga y temporada
cantidad_partidos = equipos_liga.groupby(["season", "club_id", "domestic_competition_id"]).size().reset_index(name="pj_liga")

# Comparamos con los participantes de Champions
control_liga = equipos_con_liga.merge(cantidad_partidos,on=["season", "club_id", "domestic_competition_id"],how="left")
print("EQUIPOS CON DATOS DE LIGA:",control_liga["pj_liga"].notna().sum())
print("EQUIPOS SIN DATOS DE LIGA:",control_liga["pj_liga"].isna().sum())

# Mostramos los equipos sin información
control_liga[control_liga["pj_liga"].isna()][["season", "name", "nombre_liga"]]

In [ ]:
# Seleccionamos los equipos sin datos de liga
faltantes = control_liga[control_liga["pj_liga"].isna()][["season", "club_id", "name", "domestic_competition_id"]].copy()

# Buscamos si tienen partidos en alguna otra liga
revision_faltantes = faltantes.merge(cantidad_partidos.rename(columns={"domestic_competition_id": "liga_en_partidos","pj_liga": "partidos_encontrados"}),on=["season", "club_id"],how="left")

# Mostramos todos los resultados
print(revision_faltantes.to_string(index=False))

In [ ]:
# Analizamos la cantidad de partidos registrados
# por liga y temporada
resumen_ligas = control_liga.groupby(["season", "nombre_liga"])["pj_liga"].agg(["count", "min", "max"])

# Mostramos los resultados
print("PARTIDOS REGISTRADOS POR LIGA Y TEMPORADA:")
display(resumen_ligas)

In [ ]:
# Investigamos las diferencias entre los equipos de Premier Liga
revision_premier = control_liga[(control_liga["season"] == 2021) & (control_liga["nombre_liga"] == "premier-liga")]

# Mostramos los equipos, sus ligas y sus partidos
display(revision_premier[["season", "name", "domestic_competition_id", "pj_liga"]])

In [ ]:
# Preparamos los resultados de los equipos locales
locales = partidos_liga[["season", "competition_id","home_club_id", "home_club_goals", "away_club_goals"]].copy()
locales = locales.rename(columns={"home_club_id": "club_id","home_club_goals": "goles_favor","away_club_goals": "goles_contra"})

# Preparamos los resultados de los visitantes
visitantes = partidos_liga[["season", "competition_id","away_club_id", "away_club_goals", "home_club_goals"]].copy()

visitantes = visitantes.rename(columns={"away_club_id": "club_id","away_club_goals": "goles_favor","home_club_goals": "goles_contra"})

# Unimos los dos DataFrames
resultados_liga = pd.concat([locales, visitantes], ignore_index=True)

# Mostramos las primeras cinco filas
resultados_liga.head()

In [ ]:
# Calculamos los puntos obtenidos por cada equipo
resultados_liga["puntos"] = np.select([resultados_liga["goles_favor"] > resultados_liga["goles_contra"], resultados_liga["goles_favor"] == resultados_liga["goles_contra"]],[3, 1],default=0)

# Mostramos los primeros diez resultados
resultados_liga.head(10)

In [ ]:
# Calculamos el rendimiento de cada equipo por temporada
rendimiento_liga = resultados_liga.groupby(["season", "competition_id", "club_id"]).agg(partidos_jugados=("puntos", "count"),puntos_totales=("puntos", "sum"),goles_favor=("goles_favor", "sum"),goles_contra=("goles_contra", "sum")).reset_index()

# Calculamos los puntos por partido (PPG)
rendimiento_liga["PPG"] = (rendimiento_liga["puntos_totales"] / rendimiento_liga["partidos_jugados"])

# Mostramos los primeros diez equipos
rendimiento_liga.head(10)

In [ ]:
# Preparamos el rendimiento de liga para combinarlo
rendimiento_liga_2 = rendimiento_liga.rename(columns={"competition_id": "domestic_competition_id","PPG": "PPG_liga"})

# Combinamos los equipos de Champions con su rendimiento en liga
comparacion_liga = equipos_con_liga.merge(rendimiento_liga_2,on=["season", "club_id", "domestic_competition_id"],how="left")

# Verificamos los resultados
print("TOTAL DE REGISTROS:", comparacion_liga.shape[0])
print("EQUIPOS CON RENDIMIENTO DE LIGA:",comparacion_liga["PPG_liga"].notna().sum())
print("EQUIPOS SIN RENDIMIENTO DE LIGA:",comparacion_liga["PPG_liga"].isna().sum())

# Mostramos los primeros diez equipos
comparacion_liga.head(10)

In [ ]:
# Preparamos los resultados de los equipos locales
locales_champions = champions[["season", "home_club_id", "home_club_goals", "away_club_goals"]].copy()
locales_champions = locales_champions.rename(columns={"home_club_id": "club_id", "home_club_goals": "goles_favor", "away_club_goals": "goles_contra"})

# Preparamos los resultados de los visitantes
visitantes_champions = champions[["season", "away_club_id", "away_club_goals", "home_club_goals"]].copy()
visitantes_champions = visitantes_champions.rename(columns={"away_club_id": "club_id", "away_club_goals": "goles_favor", "home_club_goals": "goles_contra"})

# Unimos ambos DataFrames
resultados_champions = pd.concat([locales_champions, visitantes_champions], ignore_index=True)

# Calculamos los puntos de cada equipo
resultados_champions["puntos"] = np.select([resultados_champions["goles_favor"] > resultados_champions["goles_contra"], resultados_champions["goles_favor"] == resultados_champions["goles_contra"]], [3, 1], default=0)

# Verificamos los resultados
print("TOTAL DE REGISTROS:", resultados_champions.shape[0])
resultados_champions.head(10)

In [ ]:
# Calculamos el rendimiento de cada equipo en Champions
rendimiento_champions = resultados_champions.groupby(["season", "club_id"]).agg(partidos_champions=("puntos", "count"), puntos_champions=("puntos", "sum"), goles_favor_champions=("goles_favor", "sum"), goles_contra_champions=("goles_contra", "sum")).reset_index()

# Calculamos los puntos por partido (PPG)
rendimiento_champions["PPG_champions"] = (rendimiento_champions["puntos_champions"] / rendimiento_champions["partidos_champions"])

# Verificamos los resultados
print("TOTAL DE REGISTROS:", rendimiento_champions.shape[0])
rendimiento_champions.head(10)